In [ ]:
#guinea merger NOT USED, SPANISH NO ADDITIONAL RESULTS

"""
Merger for Equatorial Guinea fix run.
Combines the main dataset (output_africa_BigCity) with the new search results
for Malabo and Bata (output_equatorial_guinea_BigCity), which now include Spanish.

Produces:
- all_africa_combined_final.csv / .json
- all_africa_combined_final_filling_3857.gpkg
- all_africa_combined_final_reseller_3857.gpkg
- summary_final.csv (updated summary with new city stats)
"""

import csv
import json
import os
from collections import Counter

# ==================== CONFIGURATION ====================
MAIN_DIR = "output_africa_BigCity"
FIX_DIR = "output_equatorial_guinea_BigCity"

# Timestamp parts of the files (change these according to your actual files)
MAIN_TIMESTAMP = "20260429_104943"      # main run timestamp
FIX_TIMESTAMP  = "202604XX_XXXXXX"      # fix run timestamp (insert yours)

# Centers of the two cities (used to identify old points to replace)
MALABO_CENTER = (3.750, 8.783)
BATA_CENTER   = (1.867, 9.767)
TOLERANCE = 0.001   # coordinate tolerance to match

# Output directory (we will write into MAIN_DIR)
OUTPUT_DIR = MAIN_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================== LOAD FILES ====================
main_csv = os.path.join(MAIN_DIR, f"all_africa_combined_{MAIN_TIMESTAMP}.csv")
fix_csv  = os.path.join(FIX_DIR, f"all_africa_combined_{FIX_TIMESTAMP}.csv")

# Load main rows
main_rows = []
with open(main_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        main_rows.append(row)

print(f"Loaded {len(main_rows)} points from main dataset.")

# Load fix rows
fix_rows = []
with open(fix_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        fix_rows.append(row)

print(f"Loaded {len(fix_rows)} points from Equatorial Guinea fix.")

# ==================== MERGE ====================
# Function to check if a row belongs to one of the two cities
def is_guinea_city(row, tolerance=TOLERANCE):
    try:
        lat = float(row['search_lat'])
        lon = float(row['search_lon'])
    except (ValueError, KeyError):
        return False
    if abs(lat - MALABO_CENTER[0]) < tolerance and abs(lon - MALABO_CENTER[1]) < tolerance:
        return True
    if abs(lat - BATA_CENTER[0]) < tolerance and abs(lon - BATA_CENTER[1]) < tolerance:
        return True
    return False

# Remove old points from those cities
filtered_main = [row for row in main_rows if not is_guinea_city(row)]
removed = len(main_rows) - len(filtered_main)
print(f"Removed {removed} points belonging to Malabo/Bata from main dataset.")

# Combine
merged = filtered_main + fix_rows
print(f"Merged dataset size before deduplication: {len(merged)}")

# Deduplicate by place_id (keep last occurrence, i.e., fix overrides main if any overlap)
dedup = {}
for row in merged:
    pid = row['place_id']
    dedup[pid] = row   # later rows overwrite earlier
merged_dedup = list(dedup.values())
print(f"Final merged size after deduplication: {len(merged_dedup)}")

# ==================== WRITE MERGED CSV & JSON ====================
fieldnames = ['place_id','name','address','lat','lng','types','keyword',
              'search_lat','search_lon','rating','user_ratings_total','source','language','category']

final_base = os.path.join(OUTPUT_DIR, "all_africa_combined_final")
csv_out = final_base + ".csv"
with open(csv_out, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    writer.writerows(merged_dedup)
print(f"✅ Merged CSV saved: {csv_out}")

json_out = final_base + ".json"
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(merged_dedup, f, indent=2, ensure_ascii=False)
print(f"✅ Merged JSON saved: {json_out}")

# ==================== WRITE GEOPACKAGES ====================
try:
    import geopandas as gpd
    import pandas as pd
    from shapely.geometry import Point

    keep_cols = ['place_id','name','address','types','keyword','search_lat','search_lon',
                 'rating','user_ratings_total','source','language','category']

    filling = [r for r in merged_dedup if r['category'] == 'filling']
    reseller = [r for r in merged_dedup if r['category'] != 'filling']

    if filling:
        df_f = pd.DataFrame(filling)
        df_f['geometry'] = [Point(float(r['lng']), float(r['lat'])) for _, r in df_f.iterrows()]
        gdf_f = gpd.GeoDataFrame(df_f, geometry='geometry', crs="EPSG:4326")
        gdf_f = gdf_f.to_crs("EPSG:3857")
        # Ensure lat/lon columns exist for compatibility
        gdf_f['lat'] = gdf_f.geometry.y
        gdf_f['lon'] = gdf_f.geometry.x
        gdf_f.to_file(final_base + "_filling_3857.gpkg", driver='GPKG', layer='filling')
        print(f"✅ Filling GeoPackage saved: {final_base}_filling_3857.gpkg")
    else:
        print("⚠️ No filling points found, skipping filling GeoPackage.")

    if reseller:
        df_r = pd.DataFrame(reseller)
        df_r['geometry'] = [Point(float(r['lng']), float(r['lat'])) for _, r in df_r.iterrows()]
        gdf_r = gpd.GeoDataFrame(df_r, geometry='geometry', crs="EPSG:4326")
        gdf_r = gdf_r.to_crs("EPSG:3857")
        gdf_r['lat'] = gdf_r.geometry.y
        gdf_r['lon'] = gdf_r.geometry.x
        gdf_r.to_file(final_base + "_reseller_3857.gpkg", driver='GPKG', layer='reseller')
        print(f"✅ Reseller GeoPackage saved: {final_base}_reseller_3857.gpkg")
    else:
        print("⚠️ No reseller points found, skipping reseller GeoPackage.")

except ImportError:
    print("⚠️ GeoPandas/Shapely not installed, skipping GeoPackage creation.")

# ==================== UPDATE SUMMARY CSV ====================
# Load original summary
main_summary_csv = os.path.join(MAIN_DIR, f"summary_{MAIN_TIMESTAMP}.csv")
fix_summary_csv  = os.path.join(FIX_DIR, f"summary_{FIX_TIMESTAMP}.csv")

orig_summary_rows = []
with open(main_summary_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        orig_summary_rows.append(row)

# Remove old rows for MALABO and BATA
orig_summary_filtered = [row for row in orig_summary_rows if row['City'] not in ('MALABO', 'BATA')]

# Read new summary rows for those cities from fix
new_guinea_rows = []
with open(fix_summary_csv, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['City'] in ('MALABO', 'BATA'):
            new_guinea_rows.append(row)

# Combine
updated_summary = orig_summary_filtered + new_guinea_rows

# Write updated summary
summary_out = os.path.join(OUTPUT_DIR, "summary_final.csv")
with open(summary_out, 'w', newline='', encoding='utf-8') as f:
    fieldnames = ['City', 'Country', 'Total_Points', 'Total_OSM', 'Total_Google',
                  'Total_Filling', 'Total_Reseller', 'Google_Languages_pct']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(updated_summary)
print(f"✅ Updated summary saved: {summary_out}")

# ==================== VALIDATION ====================
print("\n--- VALIDATION ---")

# 1. Expected point count: original_total - old_guinea + new_guinea
orig_total = len(main_rows)
orig_guinea = sum(1 for r in main_rows if is_guinea_city(r))
new_guinea = len(fix_rows)
expected = orig_total - orig_guinea + new_guinea
actual = len(merged_dedup)
print(f"Original total: {orig_total}")
print(f"Old Guinea points removed: {orig_guinea}")
print(f"New Guinea points added: {new_guinea}")
print(f"Expected merged total: {expected}")
print(f"Actual merged total: {actual}")
if expected == actual:
    print("✅ Point count matches expected.")
else:
    print(f"⚠️ MISMATCH: difference of {actual - expected}. Check for duplicates or missing points.")

# 2. Duplicate place_ids
place_ids = [r['place_id'] for r in merged_dedup]
dupes = [pid for pid, count in Counter(place_ids).items() if count > 1]
if dupes:
    print(f"⚠️ Duplicate place_ids found: {dupes[:10]}...")
else:
    print("✅ No duplicate place_ids.")

# 3. Verify new summary numbers match the merged CSV for Malabo & Bata
# Build city mapping: use search lat/lon -> city name (only for those centers)
def get_city_from_row(row):
    try:
        lat = float(row['search_lat'])
        lon = float(row['search_lon'])
    except:
        return None
    if abs(lat - MALABO_CENTER[0]) < TOLERANCE and abs(lon - MALABO_CENTER[1]) < TOLERANCE:
        return 'MALABO'
    if abs(lat - BATA_CENTER[0]) < TOLERANCE and abs(lon - BATA_CENTER[1]) < TOLERANCE:
        return 'BATA'
    return None

# Compute stats from merged CSV for these cities
malabo_points = [r for r in merged_dedup if get_city_from_row(r) == 'MALABO']
bata_points   = [r for r in merged_dedup if get_city_from_row(r) == 'BATA']

def city_stats(points):
    osm = sum(1 for r in points if r['source'] == 'osm')
    google = len(points) - osm
    filling = sum(1 for r in points if r['category'] == 'filling')
    reseller = len(points) - filling
    return len(points), osm, google, filling, reseller

for city_name, city_pts in [('MALABO', malabo_points), ('BATA', bata_points)]:
    total, osm, google, filling, reseller = city_stats(city_pts)
    # Find corresponding row in updated summary
    summary_row = next((r for r in updated_summary if r['City'] == city_name), None)
    if summary_row:
        s_total = int(summary_row['Total_Points'])
        s_osm = int(summary_row['Total_OSM'])
        s_google = int(summary_row['Total_Google'])
        s_filling = int(summary_row['Total_Filling'])
        s_reseller = int(summary_row['Total_Reseller'])
        if (total, osm, google, filling, reseller) == (s_total, s_osm, s_google, s_filling, s_reseller):
            print(f"✅ {city_name} stats match between CSV and summary.")
        else:
            print(f"⚠️ MISMATCH for {city_name}:")
            print(f"   CSV: total={total}, OSM={osm}, Google={google}, filling={filling}, reseller={reseller}")
            print(f"   Summary: total={s_total}, OSM={s_osm}, Google={s_google}, filling={s_filling}, reseller={s_reseller}")
    else:
        print(f"⚠️ {city_name} not found in updated summary.")

# 4. Quick check: total Google languages in fix run should contain 'es'
fix_google = [r for r in fix_rows if r['source'] == 'google']
langs = Counter(r['language'] for r in fix_google)
print(f"Languages in fix Google points: {dict(langs)}")
if 'es' in langs:
    print("✅ Spanish results present.")
else:
    print("⚠️ Spanish results absent – check API key or keywords.")

print("\nAll done.")